# InferCept：工具等待期间中断推理并恢复 KV

**面试问题：Agent 调用慢工具时，怎样安全暂停 LLM Decode、释放调度槽并恢复？**

## 回答主线

1. 工具型 Agent 的 LLM 请求会在生成工具参数后等待外部 I/O，继续占用调度槽会制造队头阻塞。
2. 中断点必须保存已提交 Token、KV 块映射、位置、采样计数器以及模型和分词器版本。
3. 恢复时可以复用快照，从下一 Token 继续，而基线会重算 Prompt 与已生成前缀。
4. 快照兼容性门禁比盲目恢复更重要，任何模型、RoPE、Tokenizer 或采样配置变化都应失效。
5. 调度收益要以节省的 Prefill/Decode Token、快照内存和尾延迟共同衡量。
6. 真实 InferCept 还要处理分布式 KV 搬运、取消、过期和多轮工具调用。

## 真实案例

六个客服 Agent 请求先生成工具调用，再等待支付、库存或 CRM 返回。每条请求给出 Prompt Token、已生成前缀、剩余回复、KV 块大小和等待时间；我们比较重算恢复与快照恢复，并故意用旧模型快照触发兼容性失败。数据是离线脱敏教学样本，只用于解释机制，不能宣称线上收益。

### 输入预览：六条被工具 I/O 打断的请求

In [1]:
requests = [  # 构造六条带真实调度字段的 Agent 请求。
    {"id": "R1", "tool": "payment.lookup", "prompt": 420, "prefix": 18, "remaining": 24, "wait_ms": 900},  # 支付查询等待较长。
    {"id": "R2", "tool": "inventory.check", "prompt": 260, "prefix": 12, "remaining": 16, "wait_ms": 180},  # 库存查询等待较短。
    {"id": "R3", "tool": "crm.get_user", "prompt": 610, "prefix": 22, "remaining": 30, "wait_ms": 1200},  # CRM 查询拥有长 Prompt。
    {"id": "R4", "tool": "refund.create", "prompt": 380, "prefix": 15, "remaining": 20, "wait_ms": 650},  # 退款写操作需要远端确认。
    {"id": "R5", "tool": "shipment.track", "prompt": 300, "prefix": 10, "remaining": 18, "wait_ms": 400},  # 物流查询中等等待。
    {"id": "R6", "tool": "risk.review", "prompt": 720, "prefix": 25, "remaining": 28, "wait_ms": 1500},  # 风险复核拥有最长上下文。
]  # 完成请求集合。
block_size = 16  # 假设每个物理 KV 块容纳十六个 Token。
print("请求  工具                 prompt  已生成  剩余  等待ms  KV块")  # 输出调度输入表头。
for request in requests:  # 逐请求计算中断时的 KV 块数量。
    blocks = (request["prompt"] + request["prefix"] + block_size - 1) // block_size  # 向上取整得到快照块数。
    print(f"{request['id']}   {request['tool']:<20} {request['prompt']:>6} {request['prefix']:>6} {request['remaining']:>5} {request['wait_ms']:>7} {blocks:>5}")  # 展示恢复成本差异。

请求  工具                 prompt  已生成  剩余  等待ms  KV块
R1   payment.lookup          420     18    24     900    28
R2   inventory.check         260     12    16     180    17
R3   crm.get_user            610     22    30    1200    40
R4   refund.create           380     15    20     650    25
R5   shipment.track          300     10    18     400    20
R6   risk.review             720     25    28    1500    47


## Baseline 基线：工具返回后重算 Prompt 和前缀

In [2]:
def restart_cost(request):  # 计算丢弃 KV 后恢复所需的模型 Token 工作量。
    recompute = request["prompt"] + request["prefix"]  # 重算完整 Prompt 与已提交生成前缀。
    total = recompute + request["remaining"]  # 再生成工具返回后的剩余回复。
    return {"recompute": recompute, "decode": request["remaining"], "total": total}  # 返回可解释成本分项。

restart_rows = [restart_cost(request) for request in requests]  # 计算六条请求的重启成本。
print("请求  重算Token  后续Decode  总工作量")  # 输出基线成本表头。
for request, row in zip(requests, restart_rows):  # 逐请求展示重算浪费。
    print(f"{request['id']} {row['recompute']:>10} {row['decode']:>10} {row['total']:>9}")  # 展示长 Prompt 请求的额外成本。
restart_total = sum(row["total"] for row in restart_rows)  # 汇总基线 Token 工作量。
print(f"重启基线总工作量={restart_total} token")  # 建立恢复方案的同口径对照。

请求  重算Token  后续Decode  总工作量
R1        438         24       462
R2        272         16       288
R3        632         30       662
R4        395         20       415
R5        310         18       328
R6        745         28       773
重启基线总工作量=2928 token


### 核心实现：版本化 KV 快照与恢复门禁

In [3]:
def create_snapshot(request, model_revision, tokenizer_hash):  # 创建可验证的中断快照。
    committed_tokens = request["prompt"] + request["prefix"]  # 只保存已经对用户或工具提交的前缀位置。
    kv_blocks = list(range((committed_tokens + block_size - 1) // block_size))  # 模拟逻辑 Token 到物理 KV 块的映射。
    return {"request_id": request["id"], "position": committed_tokens, "kv_blocks": kv_blocks, "rng_counter": request["prefix"], "model_revision": model_revision, "tokenizer_hash": tokenizer_hash, "sampling": "greedy"}  # 固化恢复所需合同。

def resume_cost(request, snapshot, runtime):  # 校验快照后计算安全恢复成本。
    compatible = snapshot["model_revision"] == runtime["model_revision"] and snapshot["tokenizer_hash"] == runtime["tokenizer_hash"] and snapshot["sampling"] == runtime["sampling"]  # 检查影响 KV 与后续 Token 的关键版本。
    if not compatible:  # 任一合同不一致时禁止盲目复用。
        return {"mode": "restart", **restart_cost(request), "reason": "snapshot-incompatible"}  # 降级到正确但更贵的重算路径。
    return {"mode": "resume", "recompute": 0, "decode": request["remaining"], "total": request["remaining"], "reason": "compatible"}  # 从已提交位置继续 Decode。

runtime = {"model_revision": "support-7b-r8", "tokenizer_hash": "tok-a91", "sampling": "greedy"}  # 定义当前 Serving 运行版本。
snapshots = [create_snapshot(request, runtime["model_revision"], runtime["tokenizer_hash"]) for request in requests]  # 在工具调用前为六条请求创建快照。
resume_rows = [resume_cost(request, snapshot, runtime) for request, snapshot in zip(requests, snapshots)]  # 工具返回后逐条恢复。
demo_snapshot = snapshots[0]  # 选择支付请求展示快照结构。
print("R1 快照：", demo_snapshot)  # 展示位置、块表、随机计数器和版本。
print("R1 恢复决策：", resume_rows[0])  # 展示兼容快照无需重算。

R1 快照： {'request_id': 'R1', 'position': 438, 'kv_blocks': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27], 'rng_counter': 18, 'model_revision': 'support-7b-r8', 'tokenizer_hash': 'tok-a91', 'sampling': 'greedy'}
R1 恢复决策： {'mode': 'resume', 'recompute': 0, 'decode': 24, 'total': 24, 'reason': 'compatible'}


## 结果解读：节省的重算量与快照占用

In [4]:
bytes_per_block = block_size * 4096 * 2 * 2  # 用 K/V、隐藏宽度和 FP16 字节估算单层单块占用。
print("请求  模式     重启工作  恢复工作  节省Token  快照块  估算KB/层")  # 输出结果表头。
for request, baseline, resumed, snapshot in zip(requests, restart_rows, resume_rows, snapshots):  # 逐请求对照两种恢复路径。
    saved = baseline["total"] - resumed["total"]  # 计算安全复用节省的 Token 工作量。
    memory_kb = len(snapshot["kv_blocks"]) * bytes_per_block / 1024  # 估算快照每层内存成本。
    print(f"{request['id']}   {resumed['mode']:<8} {baseline['total']:>8} {resumed['total']:>8} {saved:>9} {len(snapshot['kv_blocks']):>7} {memory_kb:>10.1f}")  # 展示时间与空间权衡。
resume_total = sum(row["total"] for row in resume_rows)  # 汇总快照恢复工作量。
saved_ratio = 1.0 - resume_total / restart_total  # 计算教学样本的理论 Token 工作降幅。
print(f"总工作量 {restart_total} -> {resume_total}，理论节省={saved_ratio:.1%}")  # 给出同口径汇总。
print("解读：长 Prompt、短后续回复最适合恢复；快照会占显存，所以调度器还要按等待时间和内存压力决定保留、搬移或丢弃。")  # 解释适用边界。

请求  模式     重启工作  恢复工作  节省Token  快照块  估算KB/层
R1   resume        462       24       438      28     7168.0
R2   resume        288       16       272      17     4352.0
R3   resume        662       30       632      40    10240.0
R4   resume        415       20       395      25     6400.0
R5   resume        328       18       310      20     5120.0
R6   resume        773       28       745      47    12032.0
总工作量 2928 -> 136，理论节省=95.4%
解读：长 Prompt、短后续回复最适合恢复；快照会占显存，所以调度器还要按等待时间和内存压力决定保留、搬移或丢弃。


## 失败案例：模型热更新后盲目复用旧 KV

In [5]:
stale_snapshot = create_snapshot(requests[0], "support-7b-r7", "tok-a91")  # 构造旧模型版本生成的 KV 快照。
unsafe_next_token = "退款5天" if stale_snapshot["model_revision"] != runtime["model_revision"] else "退款3天"  # 模拟忽略版本后产生的旧政策 Token。
safe_decision = resume_cost(requests[0], stale_snapshot, runtime)  # 使用兼容性门禁评估旧快照。
safe_next_token = "退款3天" if safe_decision["mode"] == "restart" else unsafe_next_token  # 降级重算后使用当前模型政策。
print(f"盲目恢复旧快照：revision={stale_snapshot['model_revision']} next_token={unsafe_next_token}")  # 展示静默错误而非崩溃。
print(f"门禁决策：{safe_decision}，安全输出={safe_next_token}")  # 展示版本失配触发重算。
print("修正策略：模型、Tokenizer、RoPE、KV dtype、采样器任一变化都使快照失效；不能只比 request_id。")  # 总结恢复兼容性。

盲目恢复旧快照：revision=support-7b-r7 next_token=退款5天
门禁决策：{'mode': 'restart', 'recompute': 438, 'decode': 24, 'total': 462, 'reason': 'snapshot-incompatible'}，安全输出=退款3天
修正策略：模型、Tokenizer、RoPE、KV dtype、采样器任一变化都使快照失效；不能只比 request_id。


### 生产边界与调度事件

In [6]:
scheduler_event = {"request_id": "R1", "event": "resume", "wait_ms": requests[0]["wait_ms"], "snapshot_blocks": len(snapshots[0]["kv_blocks"]), "compatibility": "pass", "saved_tokens": restart_rows[0]["total"] - resume_rows[0]["total"]}  # 构造可观测恢复事件。
print("调度事件：", scheduler_event)  # 展示线上评估所需字段。
print("生产替换点：真实系统需要 GPU KV 块引用计数、主机/设备换入换出、取消回收、Deadline 调度、分层缓存和故障恢复。")  # 明确列表块表与真实 InferCept 的差距。

调度事件： {'request_id': 'R1', 'event': 'resume', 'wait_ms': 900, 'snapshot_blocks': 28, 'compatibility': 'pass', 'saved_tokens': 438}
生产替换点：真实系统需要 GPU KV 块引用计数、主机/设备换入换出、取消回收、Deadline 调度、分层缓存和故障恢复。


## 回归测试：最后只保护恢复成本与兼容门禁

In [7]:
assert all(row["mode"] == "resume" and row["recompute"] == 0 for row in resume_rows)  # 验证兼容快照从已提交位置继续。
assert resume_total < restart_total and saved_ratio > 0.80  # 验证六条长 Prompt 请求显著减少重算工作。
assert demo_snapshot["position"] == requests[0]["prompt"] + requests[0]["prefix"]  # 验证快照位置覆盖已提交上下文。
assert safe_decision["mode"] == "restart" and safe_decision["reason"] == "snapshot-incompatible"  # 验证旧模型快照被拒绝。
assert safe_next_token == "退款3天" and unsafe_next_token != safe_next_token  # 验证失败探针和修正输出确实不同。
print("回归测试通过：快照位置、零重算恢复、成本节省、版本失配降级和政策输出修正均成立。")  # 用少量断言总结恢复合同。

回归测试通过：快照位置、零重算恢复、成本节省、版本失配降级和政策输出修正均成立。
